# ▶ Run the NeuroBridge website (with the agent and the AI models)

**Before you start (one time):**
1. Upload the project zip to the top of your Google Drive (*My Drive*): either `NeuroBridge_Deep_Agent_v0.6.zip`, or the repository from GitHub (**Code → Download ZIP**).
2. In Colab, click the **🔑 key icon** on the left (*Secrets*) and add these, switching **Notebook access** on for each:

| Name | What it is |
|---|---|
| `GROQ_API_KEY` | from console.groq.com → API Keys |
| `GEMINI_API_KEY` | from aistudio.google.com → Get API key |
| `TAVILY_API_KEY` | from app.tavily.com → API Keys (web search) |
| `NB_ADMIN_PASSWORD` | a password you choose (10+ characters) for the admin / moderator account |

3. **Runtime → Run all.** The last cell opens the website.

A GPU is only needed if you also want your own trained NeuroBridge models (base / SFT / SFT+ORPO) in the Model menu.
Groq and Gemini work on a normal CPU runtime.

## 1. Settings (change if you like)

In [ ]:
ADMIN_EMAIL = "admin@neurobridge.local"   # sign in with this + NB_ADMIN_PASSWORD to manage providers
LOAD_DEMO_DATA = True        # demo accounts, demo events and 4 demo clinics (all clearly labelled "demo")
USE_TRAINED_MODELS = "auto"  # "auto" = use your trained adapters if they are on Drive and a GPU is on
SEARCH_COUNTRY = "india"     # web search prefers results from this country

## 2. Get the code from Drive and install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import glob, os, zipfile, shutil
# Works with the project zip OR GitHub's "Code -> Download ZIP" of the repository.
zips = sorted(glob.glob('/content/drive/MyDrive/NeuroBridge*.zip') + glob.glob('/content/drive/MyDrive/Neurobridge*.zip'),
              key=os.path.getmtime)
assert zips, "Upload the NeuroBridge zip (or the GitHub repository zip) to My Drive first."
ZIP = zips[-1]; print("Using", ZIP)
shutil.rmtree('/content/nb', ignore_errors=True)
zipfile.ZipFile(ZIP).extractall('/content/nb')
found = glob.glob('/content/nb/**/agent/api/app.py', recursive=True)
assert found, "Could not find the source code inside the zip."
R = os.path.dirname(os.path.dirname(os.path.dirname(found[0])))
os.chdir(R); print("Source code:", R)
!pip -q install -r requirements.txt
print("Installed.")

## 3. Load your keys and check they really work

In [ ]:
from google.colab import userdata
import secrets as _s
for k in ("GROQ_API_KEY", "GEMINI_API_KEY", "TAVILY_API_KEY"):
    try:
        v = userdata.get(k)
        if v: os.environ[k] = v.strip()
    except Exception:
        pass
try:
    ADMIN_PASSWORD = userdata.get("NB_ADMIN_PASSWORD").strip()
except Exception:
    ADMIN_PASSWORD = None
if not ADMIN_PASSWORD or len(ADMIN_PASSWORD) < 10:
    ADMIN_PASSWORD = _s.token_urlsafe(12)
    print("No NB_ADMIN_PASSWORD secret (10+ characters) found - generated one for this session:")
    print("   admin password:", ADMIN_PASSWORD, "  (add it as a secret to keep it)")
os.environ["NEUROBRIDGE_SEARCH_COUNTRY"] = SEARCH_COUNTRY
!python scripts/check_keys.py

## 4. Where data is kept, admin account, optional trained models

In [ ]:
DATA = '/content/drive/MyDrive/neurobridge_data'          # accounts, journals, bookings survive restarts
os.makedirs(DATA, exist_ok=True)
os.environ.update(
    NEUROBRIDGE_APP_DB=f'{DATA}/app.sqlite3',
    NEUROBRIDGE_BOOKING_DB=f'{DATA}/bookings.sqlite3',
    NEUROBRIDGE_ADMIN_EMAIL=ADMIN_EMAIL,
    NEUROBRIDGE_ADMIN_PASSWORD=ADMIN_PASSWORD,
    NEUROBRIDGE_ADMIN_TOKEN=_s.token_urlsafe(24),
    NEUROBRIDGE_SEED_DEMO_EVENTS='1' if LOAD_DEMO_DATA else '0',
)

# Your trained models (from neurobridge_llm/NeuroBridge_LLM_Pipeline.ipynb) are used if found.
import torch
gpu = torch.cuda.is_available()
cands = sorted(glob.glob('/content/drive/MyDrive/neurobridge/llm_runs*'), key=os.path.getmtime)
sft = orpo = None
for d in reversed(cands):
    sft = sft or next(iter(glob.glob(f'{d}/sft_v2_12k*/adapter') + glob.glob(f'{d}/sft*/adapter')), None)
    orpo = orpo or next(iter(glob.glob(f'{d}/sft_orpo*/adapter')), None)
use = (USE_TRAINED_MODELS is True) or (USE_TRAINED_MODELS == "auto" and gpu and (sft or orpo))
if use:
    !pip -q install -r neurobridge_llm/requirements-train.txt
    os.environ['NEUROBRIDGE_LOCAL_MODEL'] = 'Qwen/Qwen2.5-1.5B-Instruct'
    if sft:  os.environ['NEUROBRIDGE_SFT_ADAPTER'] = sft
    if orpo: os.environ['NEUROBRIDGE_ORPO_ADAPTER'] = orpo
    print("Trained models ON:", "\n  SFT     ", sft, "\n  SFT+ORPO", orpo)
else:
    for k in ('NEUROBRIDGE_LOCAL_MODEL', 'NEUROBRIDGE_SFT_ADAPTER', 'NEUROBRIDGE_ORPO_ADAPTER'):
        os.environ.pop(k, None)
    print("Trained models off", "(no GPU)" if not gpu else "(no adapters found on Drive)" if not (sft or orpo) else "")

if LOAD_DEMO_DATA:
    DEMO_PASSWORD = 'demo-' + _s.token_hex(4)
    !python scripts/seed_demo_app.py --password "$DEMO_PASSWORD"
    print("Demo password for the three demo accounts:", DEMO_PASSWORD)

## 5. Start the server

In [ ]:
import subprocess, time, json, urllib.request, signal
try:
    server.terminate(); server.wait(5)          # restart cleanly if this cell is run again
except Exception:
    pass
for line in os.popen("pgrep -f 'uvicorn agent[.]api[.]app'").read().split():
    if int(line) != os.getpid():
        try: os.kill(int(line), signal.SIGTERM)
        except Exception: pass
log = open('/content/neurobridge_server.log', 'w')
server = subprocess.Popen(['python', '-m', 'uvicorn', 'agent.api.app:app', '--host', '0.0.0.0', '--port', '8000'],
                          stdout=log, stderr=subprocess.STDOUT, cwd=R)
for i in range(180 if use else 40):              # loading Qwen on a GPU takes ~1-2 minutes
    time.sleep(2)
    try:
        h = json.loads(urllib.request.urlopen('http://localhost:8000/health', timeout=3).read()); break
    except Exception:
        if server.poll() is not None:
            print(open('/content/neurobridge_server.log').read()[-3000:]); raise SystemExit("Server stopped - see the log above.")
else:
    raise SystemExit("Server did not start - run the troubleshooting cell at the bottom.")
print("AI models :", h['llm_providers'] or "NONE - check your keys in step 3")
print("Web search:", h['search_providers'])
if h['provider_errors']: print("Problems  :", h['provider_errors'])

## 6. Open the website

In [ ]:
from google.colab import output
print("Admin:", ADMIN_EMAIL, "/ your NB_ADMIN_PASSWORD")
if LOAD_DEMO_DATA:
    print("Demo : asha@demo.neurobridge.local (individual), ravi@demo.neurobridge.local (caregiver)  password:", DEMO_PASSWORD)
output.serve_kernel_port_as_window(8000, path='/app')

## Things to try
- **Assistant → Auto:** *"Find a psychiatrist in Panaji who understands adult ADHD and book an appointment"* — tick **Allow bookings**.
  The agent searches the web (with sources), checks the NeuroBridge directory, and asks you to approve the exact clinic, time and fee before booking.
  If no listed clinic has a time, use **＋ Track this clinic** on a source to save it with a ready-to-send message.
- **Admin → Provider directory:** add a real psychiatrist you have checked (registration number, phone call), verify it, add open times.
  Only then can people book it in NeuroBridge. Confirm or decline bookings once you hear from the clinic.
- **Community** and **Events & groups:** join groups, post, register for a support group or club.
- **Model menu** (in the Assistant) when several models are connected; **Compare** sends one message to all.
- Emergencies: *"my dad is choking"* gets fixed first-aid steps and **call 112** — no AI involved.

## Troubleshooting (run only if something is wrong)

In [ ]:
print(open('/content/neurobridge_server.log').read()[-4000:])
!python scripts/check_keys.py